# Frog Audio Clip Log

Every frog recording gets trimmed down to one usable clip before processing--that trimming happens elsewhere, and `frog_clip_log.csv` is its output. This
notebook loads the log, checks how the clips turned out, and passes the good
ones (`clips_ready`) on to the frog processing notebooks.

| Column | Description |
|---|---|
| `genus`, `species` | Taxon |
| `audio_num` | Index of the source file within the species folder (becomes `File_ID`) |
| `source_file` | Original recording |
| `clipped_file` | Saved WAV clip |
| `status` | `ok` = clipped and saved; `skipped` = no usable signal found |
| `note` | How the clip window was chosen (or why it was skipped) |
| `source_dur_s` | Duration of the original recording (s) |
| `start_time_s`, `end_time_s` | Clip bounds within the source file (s, absolute) |
| `clip_duration_s` | Duration of the saved clip (s) |

**Run order:** this notebook, then `Processing_Frog_Spectrograms.ipynb` /
`..._Multiple_Bursts.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

AUDIO_DIR = Path.home() / 'Discrete_Signals' / 'Cropped_Frogs_Audios'
NUMERIC_COLUMNS = ['audio_num', 'source_dur_s', 'start_time_s', 'end_time_s', 'clip_duration_s']

# The path columns were written on the machine that produced the clips; rewrite
# that prefix to wherever Discrete_Signals sits now.
ORIGINAL_PATH_PREFIX = '/sessions/amazing-inspiring-goodall/mnt/'
LOCAL_PATH_PREFIX = str(Path.home() / 'Discrete_Signals') + '/'

clip_log = pd.read_csv(AUDIO_DIR / 'frog_clip_log.csv')
for column in ('source_file', 'clipped_file'):
    clip_log[column] = clip_log[column].str.replace(
        ORIGINAL_PATH_PREFIX, LOCAL_PATH_PREFIX, regex=False)
for column in NUMERIC_COLUMNS:
    clip_log[column] = pd.to_numeric(clip_log[column], errors='coerce')

print(f'{len(clip_log)} rows  |  '
      f"{(clip_log['status'] == 'ok').sum()} clipped  |  "
      f"{(clip_log['status'] == 'skipped').sum()} skipped")
clip_log.head()

## How the clips were chosen

In [ ]:
clipped = clip_log[clip_log['status'] == 'ok'].copy()

# The note starts with the method name, then optional parenthetical detail
# ("burst window (2 bursts)", "fallback (whole file)", ...).
clipped['method'] = clipped['note'].str.extract(r'^([\w][\w ]+?)(?:\s*\(|$)')[0]

print('Clip method (ok rows):')
print(clipped['method'].value_counts().to_string())
print(f"\nClip duration--mean {clipped['clip_duration_s'].mean():.1f}s, "
      f"median {clipped['clip_duration_s'].median():.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

clipped['clip_duration_s'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Clip duration')
axes[0].set_xlabel('seconds')

# Source durations have a long tail (some recordings are many minutes); cap the
# axis so the bulk of the distribution is readable.
clipped['source_dur_s'].clip(upper=180).hist(bins=50, ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Source duration (capped at 180 s)')
axes[1].set_xlabel('seconds')

plt.tight_layout()
plt.show()

## Skipped recordings

In [ ]:
skipped = clip_log[clip_log['status'] == 'skipped']
print(f'{len(skipped)} recordings skipped (no usable signal found)\n')
print('Species with the most skips:')
print(skipped.groupby(['genus', 'species']).size().sort_values(ascending=False).head(20).to_string())

## Inspect one species

In [ ]:
genus, species = 'Rana', 'temporaria'          # edit to taste

match = clip_log[(clip_log['genus'] == genus) & (clip_log['species'] == species)]
match[['genus', 'species', 'audio_num', 'status',
       'start_time_s', 'end_time_s', 'clip_duration_s', 'note']]

## Handoff to the processing notebooks

In [ ]:
clips_ready = clip_log[clip_log['status'] == 'ok'][[
    'genus', 'species', 'audio_num',
    'source_file', 'clipped_file',
    'start_time_s', 'end_time_s', 'clip_duration_s', 'note',
]].reset_index(drop=True)

print(f'{len(clips_ready)} clips ready for processing')
clips_ready.head()

In [ ]:
%store clips_ready